# Entregable II completo - LogisTech

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cde571/LogisTech-Modelos-Simulacion-II/blob/main/notebooks/03_entregable_ii_completo_colab.ipynb)

Entrenamiento reproducible para regresión del tiempo de entrega y clasificación del retraso. Incluye cinco familias de modelos, validación temporal, mallas de hiperparámetros, métricas de entrenamiento/validación/prueba, intervalos de confianza, importancia de variables, PCA y UMAP.

> El modo rápido usa 15.000 pedidos distribuidos a lo largo de todo el periodo para terminar en una sesión gratuita de Colab. Cambie `MAX_ROWS = None` para ejecutar todos los pedidos; el tiempo aumentará considerablemente.

In [ ]:
from pathlib import Path
import os, subprocess, sys, urllib.request, zipfile

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Cde571/LogisTech-Modelos-Simulacion-II.git'
if IN_COLAB:
    root = Path('/content/LogisTech-Modelos-Simulacion-II')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
    os.chdir(root)
else:
    root = Path.cwd()
    if root.name == 'notebooks':
        root = root.parent
        os.chdir(root)

raw = root / 'data' / 'raw'
required = raw / 'olist_orders_dataset.csv'
if not required.exists():
    raw.mkdir(parents=True, exist_ok=True)
    archive = root / 'data' / 'olist.zip'
    urllib.request.urlretrieve('https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce', archive)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(raw)
print('Repositorio:', root)
print('CSV disponibles:', len(list(raw.glob('*.csv'))))

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from src.preprocessing import build_order_level_dataset, load_olist_tables, modeling_frame
from src.entregable_ii import ExperimentConfig, run_task

MAX_ROWS = 15_000  # Use None para la corrida completa.
config = ExperimentConfig(max_rows=MAX_ROWS, cv_splits=3, bootstrap_iterations=500)
sns.set_theme(style='whitegrid', context='notebook')
tables = load_olist_tables(raw)
orders = build_order_level_dataset(tables)
assert orders['order_id'].is_unique
print(f'Pedidos agregados: {len(orders):,}')
print(f'Periodo: {orders.order_purchase_timestamp.min()} a {orders.order_purchase_timestamp.max()}')
print('Pruebas de integridad: OK (una fila por pedido y carga completa).')

## Regresión: días hasta la entrega

In [ ]:
X_reg, y_reg, dates_reg = modeling_frame(orders, 'delivery_time_days')
reg = run_task('regression', X_reg, y_reg, dates_reg, config)
display(reg['cv_results'])
display(reg['ranking'])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=reg['ranking'], x='test_MAE', y='model', ax=ax, color='#4472C4')
ax.set(title='Regresión: MAE en prueba (menor es mejor)', xlabel='MAE [días]', ylabel='Modelo')
plt.show()
display(reg['feature_importance'])
display(reg['reductions'])

## Clasificación: probabilidad de entrega tardía

In [ ]:
X_cls, y_cls, dates_cls = modeling_frame(orders, 'late_delivery')
cls = run_task('classification', X_cls, y_cls, dates_cls, config)
display(cls['cv_results'])
display(cls['ranking'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=cls['ranking'], x='test_ROC_AUC', y='model', ax=axes[0], color='#70AD47')
axes[0].set(title='Clasificación: ROC-AUC en prueba', xlabel='ROC-AUC', ylabel='Modelo')
champion = cls['ranking'].iloc[0]['model']
champion_model = cls['models'][champion]
ConfusionMatrixDisplay.from_predictions(
    cls['parts']['y_test'], champion_model.predict(cls['parts']['X_test']),
    display_labels=['a tiempo', 'tarde'], cmap='Blues', ax=axes[1], colorbar=False
)
axes[1].set_title(f'Matriz de confusión: {champion}')
plt.tight_layout(); plt.show()
display(cls['feature_importance'])
display(cls['reductions'])

## Exportación de evidencias

Las tablas siguientes se guardan en `outputs/entregable_ii/`. La carpeta está ignorada por Git para no versionar resultados accidentales; seleccione y copie al reporte únicamente las tablas y figuras de la corrida definitiva.

In [ ]:
output = root / 'outputs' / 'entregable_ii'
output.mkdir(parents=True, exist_ok=True)
for prefix, result in [('regression', reg), ('classification', cls)]:
    for name in ['cv_results', 'ranking', 'feature_importance', 'reductions']:
        result[name].to_csv(output / f'{prefix}_{name}.csv', index=False)
summary = {
    'sample_rows': MAX_ROWS,
    'regression_champion_validation': reg['ranking'].iloc[0]['model'],
    'classification_champion_validation': cls['ranking'].iloc[0]['model'],
}
(output / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('Resultados guardados en:', output)
summary